In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\vchan\OneDrive\Desktop\mumbai_local\data\processed\timetable_with_delays.csv")
print(df.shape)
df.head()

(11730, 7)


,train_id,station,station_code,line,arrival_time,stop_number,delay_minutes
0,WR_001,Churchgate,CCG,Western,04:00,1,0
1,WR_001,Marine Lines,MEL,Western,04:03,2,1
2,WR_001,Charni Road,CYR,Western,04:05,3,1
3,WR_001,Grant Road,GTR,Western,04:08,4,1
4,WR_001,Mumbai Central,MMCT,Western,04:10,5,1


In [3]:
df['hour'] = df['arrival_time'].apply(lambda x: int(x.split(':')[0]))
df['is_peak'] = df['hour'].apply(lambda x: 1 if (8 <= x <= 10 or 18 <= x <= 20) else 0)
df['line_encoded'] = df['line'].astype('category').cat.codes
df['station_encoded'] = df['station'].astype('category').cat.codes

print(df[['hour', 'is_peak', 'line_encoded', 'station_encoded', 'delay_minutes']].head(10))

   hour  is_peak  line_encoded  station_encoded  delay_minutes
0     4        0             2               16              0
1     4        0             2               53              1
2     4        0             2               12              1
3     4        0             2               30              1
4     4        0             2               59              1
5     4        0             2               48              1
6     4        0             2               46              0
7     4        0             2               70              1
8     4        0             2               19              6
9     4        0             2               56              0


In [4]:
from sklearn.model_selection import train_test_split

In [5]:
features = ['hour', 'is_peak', 'line_encoded', 'station_encoded', 'stop_number']
target = 'delay_minutes'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 9384
Testing rows: 2346


In [6]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [7]:
model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} mins")
print(f"R2 Score: {r2:.2f}")

MAE: 1.16 mins
R2 Score: 0.82


In [9]:
import pickle

with open(r"C:\Users\vchan\OneDrive\Desktop\mumbai_local\models\delay_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved!")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\vchan\\OneDrive\\Desktop\\mumbai_local\\models\\delay_model.pkl'